<a id="top"></a>
# Marketing bancario: preparazione del dataset per il modello

La vista di partenza è una riga per **cliente × ultima chiamata**, con target binario `y` che indica se il deposito è stato sottoscritto. Qui trasformiamo questo dataset in una versione numerica, coerente e senza valori mancanti, adatta a modelli di classificazione interpretabili.

<a id="indice"></a>
## Indice

- [Obiettivo del modulo e punto di partenza](#obiettivo-modulo)
- [Caricamento del dataset base](#caricamento-dataset)
- [Tipi di variabili e primi controlli](#tipi-variabili)
- [Gestione di variabili chiave (`y`, `duration`, `pdays`)](#variabili-chiave)
- [Encoding delle variabili categoriali](#encoding-categoriche)
- [Imputazione dei valori mancanti](#imputazione-mancanti)
- [Normalizzazione delle variabili numeriche](#scaling)
- [Costruzione del dataset finale per il modello](#dataset-finale)
- [Sintesi operativa](#sintesi)
- [Primo modello di classificazione: Random Forest](#modello-rf)


<a id="obiettivo-modulo"></a>
## Obiettivo del modulo e punto di partenza

Il problema è una **classificazione binaria**: prevedere se il cliente sottoscrive il deposito (`y = 1`) oppure no (`y = 0`).

La tabella è costruita a livello di **cliente × ultima chiamata**, con variabili che arrivano da tre blocchi principali:

- anagrafica cliente;
- rischio e posizione finanziaria;
- informazioni sulla campagna corrente e sulle campagne passate.

Sul lato business l’obiettivo è ottenere un **modello interpretabile**, che aiuti la direzione marketing a capire **chi converte** e **perché**. Sul lato tecnico, questo modulo deve consegnare un dataset:

- con tipi coerenti (target numerico, feature numeriche o codificate in modo chiaro);
- senza valori mancanti in ingresso al modello;
- con un encoding delle categoriche semplice da leggere e spiegare;
- senza trasformazioni arbitrarie o difficili da giustificare.

Alcune variabili richiedono una scelta esplicita:

- `duration` misura la durata della chiamata, ma è nota **solo dopo** aver telefonato;
- `pdays = -1` non è un valore numerico reale, ma il codice “mai contattato”;
- molte variabili testuali contengono valori `yes`/`no` o un insieme ridotto di categorie.

In questa fase ci concentriamo sulla **preparazione del dato**. La modellazione vera e propria arriverà dopo, ma le decisioni prese qui condizionano direttamente quello che sarà possibile fare più avanti.


<a id="caricamento-dataset"></a>
## Caricamento del dataset base

Usiamo lo stesso dataset del modulo precedente: **Bank Marketing** dal repository UCI, caricato tramite la libreria `ucimlrepo`. Costruiamo un unico `DataFrame` con feature e target, mantenendo inalterati i nomi delle colonne originali.


In [1]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np

# Caricamento del dataset Bank Marketing (id=222 nel repository UCI)
bank_marketing = fetch_ucirepo(id=222)

X = bank_marketing.data.features
y = bank_marketing.data.targets

# Costruiamo un unico DataFrame con feature e target
df = X.copy()
if isinstance(y, pd.DataFrame):
    if y.shape[1] == 1:
        df["y"] = y.iloc[:, 0]
    else:
        for col in y.columns:
            df[col] = y[col]
else:
    df["y"] = y

print("Dimensione feature:", X.shape)
print("Dimensione target:", y.shape)
df.head()


Dimensione feature: (45211, 16)
Dimensione target: (45211, 1)


,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no


<a id="tipi-variabili"></a>
## Tipi di variabili e primi controlli

Prima di trasformare il dataset conviene verificare:

- quali colonne sono già numeriche;
- quali sono categoriali (`object` in pandas);
- dove compaiono valori mancanti o codifiche “finte” (es. `unknown`, `-1`).

Questo permette di progettare l’encoding con un minimo di disciplina, invece di accumulare trasformazioni ad hoc col passare del tempo.


In [2]:
# Tipi delle colonne
df.dtypes.to_frame("dtype")

,dtype
age,int64
job,object
marital,object
education,object
default,object
balance,int64
housing,object
loan,object
contact,object
day_of_week,int64


In [3]:
# Percentuale di valori mancanti per colonna (NaN espliciti)
df.isna().mean().sort_values(ascending=False).head(10)

poutcome     0.817478
contact      0.287983
education    0.041074
job          0.006370
month        0.000000
previous     0.000000
pdays        0.000000
campaign     0.000000
duration     0.000000
age          0.000000
dtype: float64

<a id="variabili-chiave"></a>
## Gestione di variabili chiave (`y`, `duration`, `pdays`)

Alcune colonne hanno un ruolo speciale nel task di classificazione e richiedono scelte esplicite.

### Target `y`

Il target viene fornito come stringa (`"yes"` o `"no"`). Per i modelli di classificazione servirà un **codice numerico**:

- `y = 1` se il cliente sottoscrive il deposito;
- `y = 0` altrimenti.

### `duration`: informazione post-chiamata

`duration` misura la durata dell’ultima chiamata. Dal punto di vista del modello è una variabile con **forte potere predittivo**, ma anche **avvelenata**: è nota solo dopo che la chiamata è avvenuta, quindi non è disponibile nel momento in cui bisogna decidere **chi contattare**.

Useremo `duration` solo per analisi descrittive, ma la **escluderemo esplicitamente** dal set di feature che alimenterà il modello.

<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Attenzione**

Se si lascia entrare `duration` nel modello, si rischia di costruire un sistema che sembra performare molto bene in validazione, ma che è **inutilizzabile in produzione**, perché si basa su un’informazione che in quel momento non esiste ancora.

</div>

### `pdays`: il codice “mai contattato”

La colonna `pdays` indica il numero di giorni trascorsi dall’ultima campagna in cui il cliente è stato contattato. Nel dataset UCI il valore `-1` non è un numero reale, ma il codice “mai contattato”.

Dal punto di vista del modello è più sensato lavorare così:

- creare una nuova variabile binaria `was_previously_contacted` che indica se il cliente è mai stato contattato prima;
- trasformare `pdays = -1` in valore mancante (`NaN`), da imputare in modo coerente con il resto dei dati.

In questo modo si separa l’informazione “mai contattato” dall’informazione “da quanto tempo non viene contattato”.

Infine, molte colonne (`default`, `housing`, `loan`) contengono valori `yes`/`no` e spesso anche `unknown`. Le porteremo a forma numerica trattando `unknown` come **informazione mancante** da imputare.


In [4]:
df_prep = df.copy()

# 1. Target binario y: yes/no -> 1/0
if "y" in df_prep.columns:
    df_prep["y"] = df_prep["y"].map({"yes": 1, "no": 0}).astype("int8")
    
# 2. Colonne yes/no con possibile valore 'unknown'
bool_like_cols = [col for col in ["default", "housing", "loan"] if col in df_prep.columns]

for col in bool_like_cols:
    # Trattiamo 'unknown' come mancante
    df_prep[col] = df_prep[col].replace("unknown", np.nan)
    df_prep[col] = df_prep[col].map({"yes": 1, "no": 0})

# 3. Feature engineering minimale: presenza di qualunque mutuo/prestito
#    has_any_loan = 1 se il cliente ha almeno un mutuo o un prestito personale
if set(["housing", "loan"]).issubset(df_prep.columns):
    both_unknown = df_prep[["housing", "loan"]].isna().all(axis=1)
    any_yes = (df_prep[["housing", "loan"]] == 1).any(axis=1)
    
    df_prep["has_any_loan"] = np.where(
        any_yes,
        1,
        np.where(both_unknown, np.nan, 0)
    )

# 4. Gestione di pdays e creazione di was_previously_contacted
if "pdays" in df_prep.columns:
    df_prep["was_previously_contacted"] = (df_prep["pdays"] != -1).astype("int8")
    df_prep["pdays"] = df_prep["pdays"].replace(-1, np.nan)

df_prep.head()


,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y,has_any_loan,was_previously_contacted
0,58,management,married,tertiary,0,2143,1,0,NaN,5,may,261,1,NaN,0,NaN,0,1.0,0
1,44,technician,single,secondary,0,29,1,0,NaN,5,may,151,1,NaN,0,NaN,0,1.0,0
2,33,entrepreneur,married,secondary,0,2,1,1,NaN,5,may,76,1,NaN,0,NaN,0,1.0,0
3,47,blue-collar,married,NaN,0,1506,1,0,NaN,5,may,92,1,NaN,0,NaN,0,1.0,0
4,33,NaN,single,NaN,0,1,0,0,NaN,5,may,198,1,NaN,0,NaN,0,0.0,0


<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Esercizio**

Verifica quante righe hanno `has_any_loan = 1`, quante `0` e quante `NaN`.  
Prova a spiegare a parole cosa rappresentano questi tre gruppi di clienti dal punto di vista della banca.

</div>


<a id="encoding-categoriche"></a>
## Encoding delle variabili categoriali

Per usare modelli standard di machine learning, le variabili devono essere **numeriche**. Per le categoriali distinguiamo due casi.

### Variabili ordinali

Sono quelle in cui le categorie hanno un **ordine naturale**. Nel nostro dataset l’esempio principale è `education`, che possiamo schematizzare così:

| education | interpretazione              | ordine numerico |
|----------|------------------------------|-----------------|
| primary  | istruzione primaria          | 0               |
| secondary| istruzione secondaria        | 1               |
| tertiary | laurea o superiore           | 2               |

Il codice numerico non è una misura “vera”, ma preserva l’ordine tra livelli di istruzione.

### Variabili nominali

Sono categoriali **senza ordine**: cambiare l’etichetta non cambia il significato. Ad esempio:

- `job` (tipo di lavoro);
- `marital` (stato civile);
- `contact` (tipo di contatto: telefono fisso, cellulare);
- `month` (mese della chiamata);
- `poutcome` (esito della campagna precedente).

Per queste variabili usiamo il **one-hot encoding** (o dummy encoding): per ogni categoria creiamo una colonna binaria 0/1.

Esempio con `marital`:

Tabella originale:

| id_cliente | marital  |
|-----------|----------|
| 1         | single   |
| 2         | married  |
| 3         | divorced |

Dopo il one-hot encoding (versione semplificata):

| id_cliente | marital_single | marital_married | marital_divorced |
|-----------|----------------|-----------------|------------------|
| 1         | 1              | 0               | 0                |
| 2         | 0              | 1               | 0                |
| 3         | 0              | 0               | 1                |

In pratica si scambiano etichette testuali con indicatori numerici che il modello può usare.

<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Attenzione**

Attribuire numeri a una variabile **nominale** (es. `single = 0`, `married = 1`, `divorced = 2`) è fuorviante: induce un ordine che non esiste e porta il modello a “leggere” distanze che non hanno senso.

</div>


In [5]:
df_enc = df_prep.copy()

# Variabile education come ordinale, se presente
if "education" in df_enc.columns:
    print("Valori distinti di education prima della codifica:")
    print(df_enc["education"].value_counts(dropna=False))
    
    # Definiamo un ordine semplice: primary < secondary < tertiary
    education_mapping = {
        "primary": 0,
        "secondary": 1,
        "tertiary": 2,
        "unknown": np.nan
    }
    df_enc["education"] = df_enc["education"].map(education_mapping)

# Colonne nominali da espandere con one-hot encoding
possible_nominal = ["job", "marital", "contact", "month", "poutcome"]
nominal_cols = [col for col in possible_nominal if col in df_enc.columns]

nominal_cols

Valori distinti di education prima della codifica:
education
secondary    23202
tertiary     13301
primary       6851
NaN           1857
Name: count, dtype: int64


['job', 'marital', 'contact', 'month', 'poutcome']

In [6]:
# One-hot encoding con pandas.get_dummies
# Documentazione: https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html

df_enc = pd.get_dummies(
    df_enc,
    columns=nominal_cols,
    drop_first=True  # rimuove una categoria per evitare collinearità perfetta
)

print("Forma del dataset dopo l'encoding:", df_enc.shape)
df_enc.head()


Forma del dataset dopo l'encoding: (45211, 40)


,age,education,default,balance,housing,loan,day_of_week,duration,campaign,pdays,...,month_jan,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_other,poutcome_success
0,58,2.0,0,2143,1,0,5,261,1,NaN,...,False,False,False,False,True,False,False,False,False,False
1,44,1.0,0,29,1,0,5,151,1,NaN,...,False,False,False,False,True,False,False,False,False,False
2,33,1.0,0,2,1,1,5,76,1,NaN,...,False,False,False,False,True,False,False,False,False,False
3,47,NaN,0,1506,1,0,5,92,1,NaN,...,False,False,False,False,True,False,False,False,False,False
4,33,NaN,0,1,0,0,5,198,1,NaN,...,False,False,False,False,True,False,False,False,False,False


In [7]:
# Controllo dei valori mancanti dopo l'encoding
df_enc.isna().mean().sort_values(ascending=False).head(10)

pdays                0.817367
education            0.041074
month_jan            0.000000
job_technician       0.000000
job_unemployed       0.000000
marital_married      0.000000
marital_single       0.000000
contact_telephone    0.000000
month_aug            0.000000
month_dec            0.000000
dtype: float64

<a id="imputazione-mancanti"></a>
## Imputazione dei valori mancanti

Per molti algoritmi di machine learning non è possibile avere valori mancanti nelle feature in input. Ora che tutte le colonne (escluse poche eccezioni) sono numeriche, possiamo occuparci dell’**imputazione**.

Esistono due famiglie di approcci:

- metodi semplici, come la sostituzione con **media**, **mediana** o **moda** (`sklearn.impute.SimpleImputer`);
- metodi più sofisticati, che stimano i valori mancanti usando modelli addestrati sulle altre colonne.

Useremo l’**Iterative Imputer** di scikit-learn (`sklearn.impute.IterativeImputer`), che costruisce in modo iterativo un modello per ogni colonna con missing usando le altre come predittori.

Dal punto di vista dell’API è un **trasformer**:

- si adatta ai dati con `fit`;
- applica la trasformazione con `transform` (o `fit_transform` per fare entrambi in un colpo solo).

Questo schema `fit/transform` tornerà in modo ricorrente nei moduli successivi.


In [8]:
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer

# Separiamo target e variabili candidate per il modello
y_target = df_enc["y"]
feature_cols = [col for col in df_enc.columns if col not in ["y", "duration"]]

X_features = df_enc[feature_cols]

print("Numero di feature candidate (esclusa duration):", len(feature_cols))

# Iterative Imputer
imputer = IterativeImputer(random_state=42)

X_imputed_array = imputer.fit_transform(X_features)

# Ricreiamo un DataFrame per mantenere nomi di colonne e indice
X_imputed = pd.DataFrame(X_imputed_array, columns=feature_cols, index=df_enc.index)

# Verifica che non restino valori mancanti
X_imputed.isna().sum().sum()


Numero di feature candidate (esclusa duration): 38


np.int64(0)

In [9]:
X_imputed.head()

,age,education,default,balance,housing,loan,day_of_week,campaign,pdays,previous,...,month_jan,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_other,poutcome_success
0,58.0,2.000000,0.0,2143.0,1.0,0.0,5.0,1.0,282.522908,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,44.0,1.000000,0.0,29.0,1.0,0.0,5.0,1.0,301.223156,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,33.0,1.000000,0.0,2.0,1.0,1.0,5.0,1.0,292.923531,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,47.0,0.504972,0.0,1506.0,1.0,0.0,5.0,1.0,322.665363,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,33.0,1.148689,0.0,1.0,0.0,0.0,5.0,1.0,259.679072,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


<a id="scaling"></a>
## Normalizzazione delle variabili numeriche

Alcuni modelli sono sensibili alla scala delle variabili: regressioni lineari, SVM, KNN, reti neurali.  
Altri modelli, in particolare gli alberi decisionali e i loro ensemble (Random Forest, Gradient Boosting), sono invece molto meno influenzati dalla scala perché lavorano su soglie e partizioni dello spazio delle feature.

Normalizzare non aggiunge informazione, ma aiuta a:

- evitare che una singola feature influisca troppo solo grazie alla sua scala;
- stabilizzare e velocizzare l’ottimizzazione in modelli basati su gradienti;
- rendere più confrontabili i coefficienti quando si usano modelli lineari interpretabili.

### Perché usare *RobustScaler*

In molti dataset reali compaiono code lunghe e valori estremi.  
`RobustScaler` usa **mediana** e **interquartile range (IQR)** invece di media e deviazione standard, e risulta quindi più stabile in presenza di outlier.

La normalizzazione si applica **dopo**:

1. encoding delle variabili categoriali;
2. imputazione dei valori mancanti.

### E le variabili binarie?

Le variabili già codificate come 0/1 vengono di solito **lasciate così**:

- hanno già una scala compatta e leggibile;
- scalare 0/1 non porta beneficio, anzi rende più difficile leggere i coefficienti in modelli interpretabili.

Useremo quindi lo scaling solo sulle feature numeriche non binarie.


In [10]:
from sklearn.preprocessing import RobustScaler

# Identifichiamo le colonne binarie (solo valori 0/1)
binary_cols = []
for col in X_imputed.columns:
    unique_vals = pd.unique(X_imputed[col])
    # Consideriamo binaria una colonna che contiene solo 0/1 (eventuali float tipo 0.0/1.0 sono ammessi)
    if set(unique_vals).issubset({0, 1, 0.0, 1.0}):
        binary_cols.append(col)

numeric_cols = [col for col in X_imputed.columns if col not in binary_cols]

scaler = RobustScaler()
X_scaled = X_imputed.copy()
X_scaled[numeric_cols] = scaler.fit_transform(X_imputed[numeric_cols])

print("Numero di feature binarie lasciate inalterate:", len(binary_cols))
print("Numero di feature scalate:", len(numeric_cols))

X_scaled.head()


Numero di feature binarie lasciate inalterate: 31
Numero di feature scalate: 7


,age,education,default,balance,housing,loan,day_of_week,campaign,pdays,previous,...,month_jan,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_other,poutcome_success
0,1.266667,1.000000,0.0,1.250000,1.0,0.0,-0.846154,-0.5,0.307030,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,0.333333,0.000000,0.0,-0.308997,1.0,0.0,-0.846154,-0.5,0.466139,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,-0.400000,0.000000,0.0,-0.328909,1.0,1.0,-0.846154,-0.5,0.395522,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,0.533333,-0.495028,0.0,0.780236,1.0,0.0,-0.846154,-0.5,0.648576,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,-0.400000,0.148689,0.0,-0.329646,0.0,0.0,-0.846154,-0.5,0.112667,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


<a id="dataset-finale"></a>
## Costruzione del dataset finale per il modello

Abbiamo ora:

- un blocco di feature numeriche senza valori mancanti (`X_imputed`), non normalizzate;
- una versione con scaling robusto delle sole feature non binarie (`X_scaled`);
- il target binario `y_target`;
- la variabile `duration`, che resta disponibile per analisi descrittive ma non verrà usata come input dei modelli di selezione dei clienti da contattare.

Per molti modelli ad alberi useremo direttamente `X_imputed` senza scaling.  
Per modelli sensibili alla scala potremo lavorare con `X_scaled` mantenendo la stessa struttura.

Costruiamo due `DataFrame` modellabili:

- `df_model` con le feature imputate non scalate;
- `df_model_scaled` con le feature scalate robustamente (solo numeriche non binarie).


In [11]:
# Dataset non scalato
df_model = X_imputed.copy()
df_model["y"] = y_target

# Dataset con scaling robusto (feature non binarie)
df_model_scaled = X_scaled.copy()
df_model_scaled["y"] = y_target

# Opzionale: manteniamo duration a parte per eventuali analisi,
# ma non la includiamo tra le feature del modello.
if "duration" in df_enc.columns:
    df_model["duration"] = df_enc["duration"]
    df_model_scaled["duration"] = df_enc["duration"]

print("Forma df_model (non scalato):", df_model.shape)
print("Forma df_model_scaled (scalato):", df_model_scaled.shape)
df_model.head()


Forma df_model (non scalato): (45211, 40)
Forma df_model_scaled (scalato): (45211, 40)


,age,education,default,balance,housing,loan,day_of_week,campaign,pdays,previous,...,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_other,poutcome_success,y,duration
0,58.0,2.000000,0.0,2143.0,1.0,0.0,5.0,1.0,282.522908,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,261
1,44.0,1.000000,0.0,29.0,1.0,0.0,5.0,1.0,301.223156,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,151
2,33.0,1.000000,0.0,2.0,1.0,1.0,5.0,1.0,292.923531,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,76
3,47.0,0.504972,0.0,1506.0,1.0,0.0,5.0,1.0,322.665363,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,92
4,33.0,1.148689,0.0,1.0,0.0,0.0,5.0,1.0,259.679072,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,198


<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Nota importante**

Quando passerai alla modellazione, costruisci esplicitamente la matrice delle feature **escludendo** la colonna `duration`.  
Se vuoi confrontare le prestazioni di un modello “realistico” con uno che usa anche `duration`, fallo in modo controllato e dichiarato, come esperimento metodologico, non come soluzione da portare in produzione.

</div>


<a id="sintesi"></a>
## Sintesi operativa

In questo modulo il dataset Bank Marketing è stato trasformato in una versione pronta per la modellazione supervisionata su `y` come classificazione binaria. I passaggi principali sono stati:

- normalizzare il target (`y`) e le variabili booleane (`yes`/`no`) in forma numerica;
- trattare in modo esplicito variabili delicate (`duration`, `pdays`) e creare poche feature derivate ad alta leggibilità (`has_any_loan`, `was_previously_contacted`);
- distinguere tra categoriali ordinali e nominali, usando codifica numerica ordinata per la prima (`education`) e one-hot encoding per le seconde;
- convertire `unknown` e codici speciali in valori mancanti reali (`NaN`);
- applicare un imputer multivariato (`IterativeImputer`) per ottenere una matrice di feature numeriche senza missing;
- introdurre la normalizzazione con `RobustScaler`, utile per i modelli sensibili alla scala delle feature;
- comporre i dataset finali `df_model` e `df_model_scaled`, che contengono le feature modellabili e il target binario.

Nel passo successivo cominciamo a collegare questi dataset a un primo modello supervisionato.


<a id="modello-rf"></a>
## Primo modello di classificazione: Random Forest

Per chiudere il lavoro di preparazione dati costruiamo un primo modello di riferimento.  
Usiamo una **Random Forest di classificazione** addestrata sul dataset imputato ma non normalizzato (`X_imputed`).

La scelta è intenzionale: i modelli ad alberi (decision tree, random forest, gradient boosting) non hanno bisogno di scaling, perché lavorano su soglie e partizioni dello spazio delle feature, non su distanze euclidee. Questo li rende utili come primo baseline robusto.

Lo schema è essenziale:

1. separiamo i dati in train e test;
2. addestriamo una `RandomForestClassifier` sulle feature numeriche imputate (`X_imputed`) e sul target binario (`y_target`);
3. osserviamo alcune previsioni (classe e probabilità di conversione);
4. visualizziamo la rappresentazione HTML del modello, utile per ispezionare struttura e iperparametri.

Non è ancora la fase di modellazione completa, ma è un primo collegamento concreto tra dataset preparato e modello supervisionato.


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.utils import estimator_html_repr
from IPython.display import HTML, display

# Usiamo il dataset imputato NON normalizzato
X = X_imputed.copy()
y = y_target

# Train/test split per una valutazione rapida fuori campione
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

rf_clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

rf_clf.fit(X_train, y_train)

print("Report di classificazione sul test set:\n")
y_pred = rf_clf.predict(X_test)
print(classification_report(y_test, y_pred))

# Previsioni di esempio con probabilità di conversione
y_proba = rf_clf.predict_proba(X_test)[:, 1]

preview = pd.DataFrame({
    "y_true": y_test.iloc[:10].values,
    "y_pred": y_pred[:10],
    "p_conversion": np.round(y_proba[:10], 3)
})

print("\nPrime 10 previsioni (classe e probabilità di sottoscrizione):")
display(preview)

# Rappresentazione HTML del modello (struttura e iperparametri)
HTML(estimator_html_repr(rf_clf))

Report di classificazione sul test set:

              precision    recall  f1-score   support

           0       0.91      0.98      0.94      7985
           1       0.66      0.22      0.33      1058

    accuracy                           0.90      9043
   macro avg       0.78      0.60      0.64      9043
weighted avg       0.88      0.90      0.87      9043


Prime 10 previsioni (classe e probabilità di sottoscrizione):


,y_true,y_pred,p_conversion
0,0,0,0.015
1,0,0,0.020
2,0,0,0.040
3,0,0,0.075
4,0,0,0.090
5,0,0,0.045
6,0,0,0.215
7,1,1,0.745
8,0,0,0.065
9,0,0,0.025


,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False
